# 01. Stylized facts of equity returns

Four empirical regularities, verified on ~20 years of S&P 500 daily returns,
followed by the rolling-volatility estimator that GARCH replaces in notebook 02.

Each section states the definition it uses, computes the quantity, and ends with
what the number means.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scipy
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import os
path_here = os.getcwd()
project_path = os.path.dirname(path_here)
os.chdir(project_path)

from garch_lab.data import load_prices

px = load_prices("^GSPC", start="2005-01-01")  # downloads once, then reads data/*.csv

END_DATE = "2026-08-03"  # the end date is frozen so the notebook is reproducible
px = px[px.index <= END_DATE]
px.tail()

date
2026-07-28    7428.779785
2026-07-29    7316.149902
2026-07-30    7437.629883
2026-07-31    7489.720215
2026-08-03    7556.790039
Name: ^GSPC, dtype: float64

## 1. From prices to returns

Prices are non-stationary, so statistics computed on them do not converge to
anything interpretable: the time average of the S&P's level says only how long
the sample is. Returns are approximately stationary and scale-free, which makes
them the object of study.

We use log returns, $r_t = \ln(P_t / P_{t-1})$. They are additive over time, so
the 5-day log return is exactly the sum of five daily ones. Simple returns are
additive across assets instead. Volatility modelling lives on the time axis, so
log returns are the right default here.

Implemented in [src/garch_lab/returns.py](../src/garch_lab/returns.py), specified
by `tests/test_returns.py`.

In [2]:
from garch_lab.returns import log_returns

r = log_returns(px)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Price Level", "Daily Log Returns")
)

# Top: price level
fig.add_trace(
    go.Scatter(
        x=px.index,
        y=px,
        mode="lines",
        name="Price"
    ),
    row=1,
    col=1
)

# Bottom: log returns
fig.add_trace(
    go.Scatter(
        x=r.index,
        y=r,
        mode="lines",
        name="Log Return"
    ),
    row=2,
    col=1
)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Log Return", row=2, col=1)

fig.update_layout(
    height=700,
    showlegend=False,
    title="Price and Daily Log Returns"
)

fig.show()

The returns seem to depend on the previous day's returns: the heavy-volatility
days are concentrated in the same years, 2008 and 2020. An iid model forbids
exactly that kind of clustering.

## 2.1 Fat tails

**Excess kurtosis.** $\kappa = \dfrac{E[(X-\mu)^4]}{\sigma^4} - 3$, the fourth
standardized moment minus 3 (the Gaussian's value), so $\kappa = 0$ for a normal
distribution.

**Jarque-Bera statistic.** $JB = \dfrac{n}{6}\left(S^2 + \dfrac{\kappa^2}{4}\right)$,
combining sample skewness $S$ and excess kurtosis $\kappa$; under normality
$JB \sim \chi^2_2$.

Two things to note before computing. A Gaussian would produce
$6.3 \times 10^{-5} \times 5300 < 1$ day beyond $4\hat{\sigma}$ in a sample this
size; with $\hat{\sigma} \approx 1.2\%$ that threshold is about 5%, and the sample
has many days past it. And the sampling variance of kurtosis depends on the
eighth moment, so single outliers such as 2008 and 2020 dominate it, which makes
the estimate unstable across subsamples.

Below: excess kurtosis on the full sample and on each half, the count of
$|r| > 4\hat{\sigma}$ days, a QQ plot of standardized returns against the normal,
a Student-t fit by maximum likelihood, and Jarque-Bera.

In [4]:
# (a) excess kurtosis on the full sample and on each half separately

full_kurtosis = scipy.stats.kurtosis(r)
full_skewness = scipy.stats.skew(r)
first_half_kurtosis = scipy.stats.kurtosis(r[:len(r)//2])
second_half_kurtosis = scipy.stats.kurtosis(r[len(r)//2:])

print("Full sample kurtosis:", full_kurtosis)
print("First half kurtosis:", first_half_kurtosis)
print("Second half kurtosis:", second_half_kurtosis)
print("Full sample skewness:", full_skewness)

# The two halves differ substantially.

Full sample kurtosis: 13.221577458135577
First half kurtosis: 10.961235329154114
Second half kurtosis: 16.290031048831707
Full sample skewness: -0.47611693744255623


In [5]:
# (b) count days with |r| > 4 sigma-hat

full_volatility = scipy.stats.tstd(r)

print("Full sample volatility (std dev):", full_volatility)
print("Number of days with |r| > 4σ:", r[abs(r) > 4 * full_volatility].count())
# Far more than the Gaussian's fewer-than-one.

Full sample volatility (std dev): 0.012007501820232734
Number of days with |r| > 4σ: 43


In [6]:
# (c) QQ plot of standardized returns vs the normal

r_ordered = scipy.stats.probplot(r, sparams=(scipy.stats.tmean(r), scipy.stats.tstd(r)))

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=r_ordered[0][0],
        y=r_ordered[0][1],
        mode='markers',
        name='QQ Plot'
    )
)

# Add the ideal line
min_val = min(r_ordered[0][1].min(), r_ordered[0][0].min())
max_val = max(r_ordered[0][1].max(), r_ordered[0][0].max())
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        name='Ideal Line',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="QQ plot: daily log returns vs Normal (return units)",
                   xaxis_title="Theoretical quantiles", yaxis_title="Empirical quantiles")


fig.show()

fig.write_image("figures/fig2_qq_tails.png")

In [7]:
# (d) fit a Student-t by maximum likelihood, report nu-hat
# scipy returns (df, loc, scale); scale is not sigma, since Var = scale^2 * nu/(nu-2)

df, loc, scale = scipy.stats.t.fit(r)

estimated_t_sigma = scale * np.sqrt(df / (df - 2))

print("Estimated degrees of freedom (ν):", df)
print("Estimated σ:", estimated_t_sigma)

Estimated degrees of freedom (ν): 2.433390541651497
Estimated σ: 0.015250653701340038


In [8]:
# (e) Jarque–Bera

jb = scipy.stats.jarque_bera(r)

print("Jarque-Bera test statistic:", jb.statistic)
print("Jarque-Bera p-value:", jb.pvalue)
print("The JB >> 1 and its p-value << 0.05, thus we reject the hypothesis of normality for the log returns.")

Jarque-Bera test statistic: 39741.29646578006
Jarque-Bera p-value: 0.0
The JB >> 1 and its p-value << 0.05, thus we reject the hypothesis of normality for the log returns.


The Jarque-Bera p-value is nearly uninformative here because the sample is large
enough that even small deviations from normality are rejected. The information is
carried instead by the effect sizes: the kurtosis and skewness values, which show
heavy tails and asymmetry, and the instability of kurtosis between the two halves,
which is itself a finding about the tails rather than a bug.

## 2.2 No linear autocorrelation in returns

**Autocorrelation function (ACF).** For a weakly stationary series $\{X_t\}$ with
mean $\mu = E[X_t]$ and variance $\gamma_0 = \mathrm{Var}(X_t)$, the autocorrelation
at lag $k$ is

$$
\rho(k) = \frac{\gamma_k}{\gamma_0}, \qquad \gamma_k = \mathrm{Cov}(X_t, X_{t+k}) = E\big[(X_t - \mu)(X_{t+k} - \mu)\big].
$$

The sample estimator, for a series of length $n$, is

$$
\hat{\rho}(k) = \frac{\sum_{t=1}^{n-k} (X_t - \bar{X})(X_{t+k} - \bar{X})}{\sum_{t=1}^{n} (X_t - \bar{X})^2}.
$$

Under the null that $\{X_t\}$ is iid, $\hat{\rho}(k)$ is asymptotically
$\mathcal{N}(0, 1/n)$ for each fixed $k > 0$, which is where the
$\pm 1.96/\sqrt{n}$ band comes from. That band assumes iid, and heteroskedasticity
inflates the true sampling variance of $\hat{\rho}$, so marginal violations should
not be over-read.

Strong linear predictability in returns would be a free lunch, and trading it away
removes it. What survives sits at or below the noise floor.

In [9]:
import statsmodels.api as sm

acf = sm.tsa.stattools.acf(r, nlags=50)[1:]  # exclude lag 0, which is 1.0 by definition

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=np.arange(1, len(acf)+1),
        y=acf,
        name='ACF'
    )
)

# 95% confidence band under the iid null
conf_band = 1.96/np.sqrt(len(r))
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(acf)+1),
        y=[conf_band]*len(acf),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(acf)+1),
        y=[-conf_band]*len(acf),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="ACF of daily log returns",
                   xaxis_title="Lag", yaxis_title="ACF")


fig.show()

Most lags sit inside the band. The lag-1 autocorrelation is the exception: it
falls well outside, and I have not explained why.

Uncorrelated is not independent. Correlation is a first-order property, so
second- and higher-order structure can survive untouched, which is where the next
section looks.

## 2.3 Volatility clustering

The series $r$ and $r^2$ carry the same digits but not the same information: the
sign is absent from $r^2$. That is the hint that magnitude is predictable while
sign is not, and it is what the decomposition $r_t = \sigma_t z_t$ formalizes,
with $z_t$ carrying the unpredictable sign and all the memory living in
$\sigma_t$.

Below: the ACF of $r^2$ and of $|r|$ out to lag 100, exported as
`figures/fig1_clustering.png`.

In [10]:
acf_r2 = sm.tsa.stattools.acf(r**2, nlags=100)[1:]      # exclude lag 0
acf_abs_r = sm.tsa.stattools.acf(np.abs(r), nlags=100)[1:]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=np.arange(1, len(acf_r2)+1),
        y=acf_r2,
        name='ACF r2'
    )
)

fig.add_trace(
    go.Bar(
        x=np.arange(1, len(acf_abs_r)+1),
        y=acf_abs_r,
        name='ACF |r|'
    )
)

# 95% confidence band under the iid null
conf_band = 1.96/np.sqrt(len(r))
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(acf_r2)+1),
        y=[conf_band]*len(acf_r2),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="ACF of daily log returns squared and absolute value",
                   xaxis_title="Lag", yaxis_title="ACF")

fig.show()

fig.write_image("figures/fig1_clustering.png")

Today's returns are similar in size to tomorrow's. Only the sign is
unpredictable.

This is the empirical license for the rest of the project: the second moment is
forecastable, which is what makes a conditional-variance model worth fitting.

## 2.4 The leverage effect

GARCH feeds shocks in as $\varepsilon^2$, which destroys the sign. The
correlation below measures whether the destroyed information matters:
$\mathrm{Corr}(r_t, |r_{t+k}|)$ for $k = 1 \dots 20$.

In [11]:
correlation_r_abs_r = [r.corr(r.abs().shift(-k)) for k in range(1,21)]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=np.arange(1, len(correlation_r_abs_r)+1),
        y=correlation_r_abs_r,
        name='Correlation'
    )
)

# 95% confidence band under the iid null
conf_band = 1.96/np.sqrt(len(r))
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(correlation_r_abs_r)+1),
        y=[-conf_band]*len(correlation_r_abs_r),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="Correlation of daily log returns with their advanced absolute values",
                   xaxis_title="Days ahead", yaxis_title="Correlation")


fig.show()

The correlation is negative at small $k$ and fades as $k$ grows. The sign of the
returns affects tomorrow's volatility differently, and a symmetric model fails to
take that into account.

This is, measured in advance, the documented failure mode of the model fitted in
notebook 02.

## 2.5 Aggregational Gaussianity

The central limit theorem pushes aggregated returns toward the Gaussian. Two
features documented above slow it down: the dependence through $\sigma_t$ and the
tail weight.

Below: excess kurtosis at 1-day, 5-day and 21-day horizons. Summing log returns
over blocks is licensed by the time-additivity tested in section 1.

In [12]:
for days in [1, 5, 21]:
    r_agg = r.rolling(window=days).sum().dropna()
    kurt = scipy.stats.kurtosis(r_agg)
    print(f"Excess kurtosis for {days}-day aggregated returns: {kurt}")

Excess kurtosis for 1-day aggregated returns: 13.221577458135577
Excess kurtosis for 5-day aggregated returns: 8.727043590032098
Excess kurtosis for 21-day aggregated returns: 9.234983035160326


Heavy tails and volatility clustering violate the CLT's independence assumption,
which slows convergence toward a Gaussian.

## 3. Rolling realized volatility

**Rolling volatility estimator.**
$\hat{\sigma}_t = \sqrt{\dfrac{1}{w}\sum_{i=0}^{w-1} r_{t-i}^2} \times \sqrt{252}$,
the annualized root-mean-square of the last $w$ squared returns, which is a
boxcar filter on $r^2$. Note that the sum runs from $i = 0$, so the estimate at
$t$ includes $r_t$.

Three conventions. The window length trades responsiveness against noise: a small
$w$ reacts fast but is jumpy, a large $w$ is smooth but stale, and $w = 21$ is one
trading month. The $\sqrt{252}$ comes from variances adding over uncorrelated
periods, which section 2.2 established, so $\sigma$ scales with the square root of
the horizon. The mean is dropped because the daily mean return is about two orders
of magnitude below daily $\sigma$, so estimating it adds more noise than the
correction removes.

Implemented in [src/garch_lab/volatility.py](../src/garch_lab/volatility.py),
specified by `tests/test_volatility.py`.

On naming: this is a 21-day rolling volatility, not realized volatility. In the
literature RV means intraday sums, which are a far less noisy estimator.

In [13]:
from garch_lab.volatility import rolling_vol

vol_d = [rolling_vol(r, window=window, annualize=True) for window in [5, 21, 63]]

# The three rolling volatilities on the same figure
fig = go.Figure()

for window, vol in zip([5, 21, 63], vol_d):
    fig.add_trace(
        go.Scatter(
            x=vol.index,
            y=vol,
            mode='lines',
            name=f'{window}-day rolling volatility'
        )
    )

fig.update_layout(title="Rolling Volatility of Daily Log Returns",
                   xaxis_title="Date", yaxis_title="Annualized Volatility")

fig.show()

fig.write_image("figures/fig3_rolling_vol.png")

Of the three windows, the 21-day one is what I would hand a risk desk: it
balances responsiveness to market changes against stability. The trade-off is
that it lags sudden shifts compared with the 5-day window.

## What the data demand of a model

1. Volatility clusters. Heavy days concentrate in the same years, 2008 and 2020,
   which an iid model forbids.
2. Returns are far from Gaussian, and the Jarque-Bera p-value is uninformative at
   this sample size. The effect sizes carry the information, and the instability
   of kurtosis across subsamples is itself a statement about the tails.
3. Linear autocorrelation in returns is essentially absent, but uncorrelated is
   not independent: correlation is only a first-order property.
4. The ACF of $|r|$ stays positive far out, so the second moment is forecastable.
   This is the license for everything in notebook 02.
5. The leverage effect is present: the sign of today's return changes tomorrow's
   volatility asymmetrically, and a symmetric model cannot reproduce it.
6. Aggregation thins the tails only slowly, because the dependence and the tail
   weight both work against the CLT.
7. A 21-day rolling volatility is the workable estimator, trading responsiveness
   for stability.

The boxcar filter above weights day 21 fully and day 22 at zero, and its window
was chosen by convention rather than from the data. Notebook 02 replaces it with
an exponentially decaying filter whose decay rate is estimated by maximum
likelihood, plus mean reversion.